In [1]:
import os
import sys
from sklearn import tree
import pandas as pd

In [2]:
student_por = "/home/ziuteng/Python_AI_for_beginner/student-por.csv"
student_por_csv = "/home/ziuteng/Python_AI_for_beginner/student_por_test.csv"
student_por_csv_1 = "/home/ziuteng/Python_AI_for_beginner/student_por_test_1.csv"
student_por_csv_2 = "/home/ziuteng/Python_AI_for_beginner/student_por_test_dummies.csv"

def save_data_frame_to_csv(data_frame:pd.DataFrame, filename:str) ->None:
    data_frame.to_csv(filename)

def load_dataset(path_of_file: str):
    """load data set (ex student portuguese scores)
    using pandas
    Args:
        path_of_file (str): path of file need to load data
    Return:
        d: data set
    """
    d = pd.read_csv(path_of_file, sep=";")
    return d

def rule_decision_tree(d: pd.DataFrame):
    d['pass'] = d.apply(lambda row: 1 if (row['G1'] + row['G2'] + row['G3'] >= 35)  else 0, axis=1)
    d = d.drop(columns=['G1', 'G2', 'G3'], axis =1) # using drop to remove column have tag G1, G2 and G3 with syntax look like this
    return d

In [3]:
d = load_dataset(student_por)

In [4]:
save_data_frame_to_csv(d, student_por_csv)
d = rule_decision_tree(d)

In [5]:
dummies_student_set = ['sex', 'school', 'address', 'famsize', 'Pstatus', 'Mjob', 'Fjob', 'reason', 'guardian', 'schoolsup', 'famsup', 'paid', 'activities',
                       'nursery', 'higher', 'internet', 'romantic']
d_test_dummies = pd.get_dummies(d, columns=dummies_student_set)
save_data_frame_to_csv(d_test_dummies, student_por_csv_2)


In [6]:
d_test_dummies = d_test_dummies.sample(frac=1)
d_train = d_test_dummies[:500]
d_test = d_test_dummies[500:]

d_train_pass = d_train['pass']
d_train_att = d_train.drop(columns=['pass'], axis=1)

d_test_pass = d_test['pass']
d_test_att = d_test.drop(columns=['pass'], axis=1)

d_pass = d_test_dummies['pass']
d_att = d_test_dummies.drop(columns=['pass'], axis=1)
import numpy as np

print("Passing: {} out of {} ({}%)".format(np.sum(d_pass), len(d_pass),round(100*(np.sum(d_pass)/len(d_pass)), 2)))

Passing: 328 out of 649 (50.54%)


In [7]:
t = tree.DecisionTreeClassifier(criterion="entropy", max_depth=5)
# khởi tạo mô hình, tạo một đối tượng cây quyết định
# criterion="entropy", chọn tiêu chí để đánh giá chất lượng phân chia, ở đây entropy có nghĩa là sử dụng thông tin entroy
# max_depth: giới hạn chiều sâu tối đa của cây quyết định là 6, giúp tránh hiện tượng overfitting
# fit là phương thức huấn luyện mô hình cây quyết định dựa trên dữ liệu đầu vào,
# d_train_att là dữ liệu đầu vào các thuộc tính
# d_train_pass là nhãn đầu ra (kết quả phân loại)
t = t.fit(d_train_att, d_train_pass)
tree.export_graphviz(t, out_file="student_per.dot", label="all", impurity=False, proportion=True, feature_names=list(d_train_att), class_names=["fail", "pass"], filled=True, rounded=True)

In [8]:
t.score(d_test_att, d_test_pass)
import pydot

def convert_dot_to_png(dot_file, png_file):
    # Load the .dot file
    (graph,) = pydot.graph_from_dot_file(dot_file)

    # Save it as a PNG file
    graph.write_png(png_file)

# Specify the input and output file names
dot_file = '/home/ziuteng/Python_AI_for_beginner/student_per.dot'  # Replace with your .dot file name
png_file = 'output_graph_6.png'  # Desired output .png file name

# Convert the file
convert_dot_to_png(dot_file, png_file)

